# ARC-AGI-3 — No-Prior Competitive Solver Agent
**NINE1EIGHT submission notebook — ARC Prize 2026 Kaggle competition**

This notebook runs a **no-prior** agent: no pretrained LLM, no external
weights, no dataset-derived priors, zero internet calls at inference time.
Every decision is derived at run time from the structure of frames actually
observed in the environment being played.

Core technique — **directed state-graph exploration over observed frames**,
the same family of approach that produced the strongest known non-LLM
result on the ARC-AGI-3 preview leaderboard (graph-based exploration over
observed frames beat every frontier LLM agent tested, at a fraction of the
token/compute cost, because average episodes run hundreds of steps and a
structural approach doesn't pay per-step LLM inference cost).

Components (all implemented in `no_prior_solver.py`, imported below):
- `FrameCodec` — exact-hash state deduplication over the 64x64, 16-colour frame stack
- `StructuralPressureScorer` — pixel/histogram/connected-component novelty scoring (also drives ACTION6 click targeting)
- `StateGraph` — directed multigraph of observed transitions, tracks per-state untried actions
- `GoExploreArchive` — "archive, restore, explore" cell selection favouring under-visited, high-pressure states
- `NoPriorSolverAgent` — orchestrates the above into the real `act(frame) -> GameAction, data` contract

Verified against the real installed `arcengine` (0.9.x) and `arc-agi` (0.9.9) packages — not assumed from memory.

## 1. Environment setup
Kaggle's accelerated (T4/P100/RTX6000) sessions run with **no internet**. `arc-agi` / `arcengine` must already be present in the kernel's environment (as provided by the official ARC-AGI-3 Kaggle starter kit) — this cell is a no-op if so, and only attempts installation when running locally with internet for development/iteration.

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec("arc_agi") is None or importlib.util.find_spec("arcengine") is None:
    # Only reached in a local dev session with internet; Kaggle's scored
    # accelerated runtime has internet disabled and must already ship
    # these packages (per the official ARC-AGI-3 Kaggle Starter kit).
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "arc-agi"], check=True)

import arc_agi
from arcengine import GameAction, GameState
print("arc_agi:", arc_agi.__file__)


## 2. No-prior solver core
Inline copy of `no_prior_solver.py` (paste-safe for a single-file Kaggle kernel submission). Keep this in sync with the canonical module in the toolkit repo.

In [ ]:
"""
no_prior_solver.py
===================
A "no prior" competitive agent for the ARC-AGI-3 track (ARC Prize 2026, Kaggle).

"No prior" means: no pretrained language model, no external weights, no
internet calls, no dataset-derived priors of any kind. Every decision the
agent makes is derived at run time from the structure of the frames it has
actually observed in *this* environment. This satisfies the competition's
hard offline-scoring constraint (Kaggle accelerated sessions run with no
internet access) and is also, independently, how the strongest public
ARC-AGI-3 preview leaderboard entries are built: directed state-graph
exploration over observed frames, not model-based guessing
(cf. "Blind Squirrel" — directed state graphs built from observed frames,
3rd place on the preview leaderboard with graph-based exploration).

Architecture
------------
1. FrameCodec         - canonical hashing / featurisation of a FrameDataRaw
                        frame stack (list[np.ndarray], each HxW, values are
                        small integer colour ids). No resizing/model needed:
                        ARC-AGI-3 frames are fixed 64x64, <=16 colours.
2. StructuralPressure  - a pure-structure "interestingness" scorer: pixel
                        delta mass, colour-histogram delta, connected-
                        component delta, border/edge activity. This is the
                        no-prior stand-in for a learned novelty/value
                        function, reused across both exploration control
                        (StructuralPressureSolver lineage) and complex-action
                        (ACTION6) coordinate targeting.
3. StateGraph          - directed multigraph over frame hashes. Nodes track
                        which of the game's *currently legal* actions have
                        been tried from them (available_actions is per-frame
                        and must be respected exactly, or ACTION6 without
                        clamped x/y will 422 against the real API).
4. GoExploreArchive     - "return-to-cell, explore-further" archive keyed by
                        frame hash, storing the shortest known action
                        trajectory from RESET that reaches each cell and a
                        visit counter, used for cell-selection probability
                        (rarely-visited cells are prioritised, matching the
                        Go-Explore "archive" mechanism).
5. NoPriorSolverAgent  - orchestrates the above into a single
                        `act(frame) -> (GameAction, dict | None)` step
                        function with RESET / WIN / GAME_OVER handling,
                        loop/no-op detection, and frontier-directed
                        exploration when no cached trajectory is more
                        promising than fresh exploration.

Verified against the real installed packages:
  - arcengine 0.9.x  (GameAction, FrameDataRaw, GameState, ComplexAction)
  - arc-agi 0.9.9    (arc_agi.Arcade, EnvironmentWrapper.action_space /
                     observation_space / step / reset / get_scorecard)

Known correct API usage (previously verified bug fix, preserved here):
  GameAction.from_id(n)   -> correct
  GameAction(n)            -> raises, NOT how the enum is constructed

Zero placeholders. Zero mock/toy code. Every function below is a real,
runnable implementation.
"""

from __future__ import annotations

import hashlib
import random
from collections import defaultdict, deque
from dataclasses import dataclass, field
from typing import Any, Optional

import numpy as np

from arcengine import FrameDataRaw, GameAction, GameState

# --------------------------------------------------------------------------
# Constants (verified: ARC-AGI-3 frames are 64x64, colour ids 0-15)
# --------------------------------------------------------------------------
GRID_SIZE = 64
MAX_COLOR = 16
COMPLEX_ACTIONS = {GameAction.ACTION6}  # only ACTION6 is_complex() in arcengine


# ==========================================================================
# 1. FrameCodec
# ==========================================================================
class FrameCodec:
    """Canonicalises a FrameDataRaw frame stack into a hashable, comparable
    representation with zero learned components.
    """

    @staticmethod
    def stack_to_array(frame: list[np.ndarray]) -> np.ndarray:
        """Stack the frame's layers into a single (L, H, W) int8 array.

        FrameDataRaw.frame is `list[np.ndarray]`; different games expose a
        different number of layers (e.g. background + foreground + overlay),
        so we do not assume a fixed layer count, only a fixed HxW per layer.
        """
        if not frame:
            return np.zeros((1, GRID_SIZE, GRID_SIZE), dtype=np.int8)
        layers = [np.asarray(layer, dtype=np.int8) for layer in frame]
        return np.stack(layers, axis=0)

    @staticmethod
    def exact_hash(arr: np.ndarray) -> str:
        """Deterministic content hash used as the state-graph node key.

        ARC-AGI-3 environments are deterministic given a fixed action
        sequence from RESET, so exact hashing (not a perceptual/fuzzy hash)
        is the correct choice for state-graph deduplication: two identical
        hashes really are the same game state.
        """
        return hashlib.blake2b(arr.tobytes(), digest_size=16).hexdigest()

    @staticmethod
    def color_histogram(arr: np.ndarray) -> np.ndarray:
        flat = arr.reshape(-1)
        hist = np.bincount(np.clip(flat, 0, MAX_COLOR - 1), minlength=MAX_COLOR)
        return hist.astype(np.float64)

    @staticmethod
    def connected_components(layer: np.ndarray) -> int:
        """4-connectivity flood-fill component count per distinct colour,
        implemented with an explicit stack (no scipy dependency required,
        keeping this Kaggle-safe under the no-internet accelerated runtime).
        """
        h, w = layer.shape
        visited = np.zeros_like(layer, dtype=bool)
        count = 0
        for y0 in range(h):
            for x0 in range(w):
                if visited[y0, x0]:
                    continue
                color = layer[y0, x0]
                stack = [(y0, x0)]
                visited[y0, x0] = True
                while stack:
                    y, x = stack.pop()
                    for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                        ny, nx = y + dy, x + dx
                        if 0 <= ny < h and 0 <= nx < w and not visited[ny, nx]:
                            if layer[ny, nx] == color:
                                visited[ny, nx] = True
                                stack.append((ny, nx))
                count += 1
        return count


# ==========================================================================
# 2. StructuralPressure — no-prior novelty / targeting scorer
# ==========================================================================
class StructuralPressureScorer:
    """Scores how "structurally interesting" a transition or a candidate
    click coordinate is, using only pixel-level structure — no learned
    weights, no pretrained embeddings.
    """

    def __init__(self, component_sample_stride: int = 4) -> None:
        # Full-resolution connected-component counting is O(HW); for the
        # per-step transition score we only need it on the *changed* layer
        # of a 64x64 grid, which is cheap, but we allow striding for very
        # hot loops (e.g. inner Go-Explore rollouts).
        self.stride = component_sample_stride

    def transition_pressure(self, prev: Optional[np.ndarray], curr: np.ndarray) -> float:
        """Higher = more structurally novel transition. Combines:
        - pixel delta mass (how much of the grid changed)
        - colour histogram delta (L1 distance)
        - connected-component delta on the primary layer
        """
        if prev is None:
            return 1.0  # first observation is maximally interesting

        if prev.shape != curr.shape:
            return 1.0  # layer-count change is itself a strong signal

        pixel_delta = float(np.mean(prev != curr))

        hist_prev = FrameCodec.color_histogram(prev)
        hist_curr = FrameCodec.color_histogram(curr)
        total_prev = hist_prev.sum() or 1.0
        total_curr = hist_curr.sum() or 1.0
        hist_delta = float(
            np.abs(hist_prev / total_prev - hist_curr / total_curr).sum()
        ) / 2.0  # normalised to [0, 1]

        cc_prev = FrameCodec.connected_components(prev[0])
        cc_curr = FrameCodec.connected_components(curr[0])
        cc_delta = abs(cc_curr - cc_prev) / max(cc_prev, cc_curr, 1)

        # Weighted sum; pixel_delta dominates because it is the most direct
        # signal that an action actually did something (vs. a no-op).
        return 0.5 * pixel_delta + 0.3 * hist_delta + 0.2 * cc_delta

    def click_target_map(self, layer: np.ndarray) -> np.ndarray:
        """Produce a (H, W) "click interestingness" map for ACTION6 targeting.

        No-prior heuristic: boundary pixels (where a cell's 4-neighbourhood
        contains more than one colour) are the structurally active parts of
        the grid — object edges, UI buttons, moving-sprite silhouettes — and
        are consistently the productive click targets in ARC-AGI-3 games
        observed so far (games are built from discrete coloured objects on
        a flat background).
        """
        h, w = layer.shape
        score = np.zeros((h, w), dtype=np.float64)
        padded = np.pad(layer, 1, mode="edge")
        center = padded[1:-1, 1:-1]
        for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            neighbor = padded[1 + dy : 1 + dy + h, 1 + dx : 1 + dx + w]
            score += (neighbor != center).astype(np.float64)
        return score  # 0..4 per cell; 0 = flat interior/background

    def best_click(
        self, layer: np.ndarray, tried: set[tuple[int, int]]
    ) -> tuple[int, int]:
        """Pick the highest-scoring untried boundary coordinate; falls back
        to the global argmax, then to a random legal coordinate if every
        boundary pixel has already been tried (keeps the agent from ever
        stalling on ACTION6-only games).
        """
        score = self.click_target_map(layer)
        order = np.dstack(np.unravel_index(np.argsort(-score.ravel()), score.shape))[0]
        for y, x in order:
            coord = (int(y), int(x))
            if coord not in tried and score[coord] > 0:
                return coord
        if order.size:
            y, x = order[0]
            return int(y), int(x)
        return random.randrange(GRID_SIZE), random.randrange(GRID_SIZE)


# ==========================================================================
# 3. StateGraph — directed multigraph over observed frame hashes
# ==========================================================================
@dataclass
class StateNode:
    frame_hash: str
    visits: int = 0
    tried_actions: set[int] = field(default_factory=set)
    tried_clicks: set[tuple[int, int]] = field(default_factory=set)
    layer_sample: Optional[np.ndarray] = None  # kept for click-target reuse


class StateGraph:
    """Directed graph: node = frame hash. Edge (node, action_id) -> next
    frame hash. This is the core of the "directed state graph exploration"
    approach that produced the strongest known preview-leaderboard result
    for ARC-AGI-3 without any model weights.
    """

    def __init__(self) -> None:
        self.nodes: dict[str, StateNode] = {}
        self.edges: dict[tuple[str, int], str] = {}
        self.reverse_edges: dict[str, list[tuple[str, int]]] = defaultdict(list)

    def get_or_create(self, frame_hash: str, layer_sample: np.ndarray) -> StateNode:
        node = self.nodes.get(frame_hash)
        if node is None:
            node = StateNode(frame_hash=frame_hash, layer_sample=layer_sample)
            self.nodes[frame_hash] = node
        node.visits += 1
        return node

    def record_edge(self, src_hash: str, action_id: int, dst_hash: str) -> None:
        self.edges[(src_hash, action_id)] = dst_hash
        self.reverse_edges[dst_hash].append((src_hash, action_id))

    def untried_actions(self, node: StateNode, legal_actions: list[int]) -> list[int]:
        return [a for a in legal_actions if a not in node.tried_actions]

    def shortest_path(self, src_hash: str, dst_hash: str) -> Optional[list[int]]:
        """BFS over recorded edges from src to dst; returns an action-id
        path, or None if dst is not (yet) reachable via known edges.
        """
        if src_hash == dst_hash:
            return []
        forward: dict[str, list[tuple[str, int]]] = defaultdict(list)
        for (s, a), d in self.edges.items():
            forward[s].append((d, a))
        queue: deque[str] = deque([src_hash])
        came_from: dict[str, tuple[str, int]] = {}
        visited = {src_hash}
        while queue:
            cur = queue.popleft()
            if cur == dst_hash:
                path: list[int] = []
                node = cur
                while node != src_hash:
                    prev, action = came_from[node]
                    path.append(action)
                    node = prev
                path.reverse()
                return path
            for nxt, action in forward.get(cur, []):
                if nxt not in visited:
                    visited.add(nxt)
                    came_from[nxt] = (cur, action)
                    queue.append(nxt)
        return None


# ==========================================================================
# 4. GoExploreArchive — cell selection favouring under-visited states
# ==========================================================================
@dataclass
class ArchiveCell:
    frame_hash: str
    trajectory: list[Any]  # sequence of (action_id, data|None) from RESET
    visits: int = 1
    pressure: float = 0.0


class GoExploreArchive:
    """Implements the Go-Explore "archive, restore, explore" loop:
    keep the shortest/best trajectory that reaches each distinct state,
    and preferentially resume exploration from states that are both
    rarely visited and structurally interesting, rather than always
    exploring forward from the current frontier (which starves the
    agent of coverage on games with early bottlenecks).
    """

    def __init__(self, rng: Optional[random.Random] = None) -> None:
        self.cells: dict[str, ArchiveCell] = {}
        self.rng = rng or random.Random(0)

    def observe(
        self, frame_hash: str, trajectory: list[Any], pressure: float
    ) -> None:
        cell = self.cells.get(frame_hash)
        if cell is None:
            self.cells[frame_hash] = ArchiveCell(
                frame_hash=frame_hash, trajectory=list(trajectory), pressure=pressure
            )
            return
        cell.visits += 1
        cell.pressure = max(cell.pressure, pressure)
        if len(trajectory) < len(cell.trajectory):
            cell.trajectory = list(trajectory)  # keep the shortest known path

    def select_restore_target(self) -> Optional[ArchiveCell]:
        """Sample a cell to return to, weighted towards low-visit-count,
        high-pressure cells (classic Go-Explore selection heuristic,
        implemented here as an explicit weighted sample rather than a
        black-box policy).
        """
        if not self.cells:
            return None
        cells = list(self.cells.values())
        weights = [
            (c.pressure + 0.05) / (c.visits ** 0.5) for c in cells
        ]
        total = sum(weights)
        if total <= 0:
            return self.rng.choice(cells)
        r = self.rng.uniform(0, total)
        acc = 0.0
        for c, w in zip(cells, weights):
            acc += w
            if acc >= r:
                return c
        return cells[-1]


# ==========================================================================
# 5. NoPriorSolverAgent — main agent
# ==========================================================================
class NoPriorSolverAgent:
    """Single-file, zero-dependency-on-external-weights agent implementing
    the standard ARC-AGI-3-Agents `act(frame) -> GameAction[, data]`
    contract, driven entirely by StateGraph + GoExploreArchive +
    StructuralPressureScorer.

    Usage (matches the real arc_agi.Arcade / arcengine API):

        from arc_agi import Arcade
        from arcengine import GameAction

        arc = Arcade()
        env = arc.make("ls20")
        agent = NoPriorSolverAgent()

        frame = env.reset()
        while True:
            action, data = agent.act(frame, legal_actions=env.action_space)
            frame = env.step(action, data=data)
            if frame is None:
                break
            if frame.state.name == "WIN":
                break
    """

    def __init__(self, seed: int = 0, loop_window: int = 12) -> None:
        self.rng = random.Random(seed)
        self.codec = FrameCodec()
        self.pressure = StructuralPressureScorer()
        self.graph = StateGraph()
        self.archive = GoExploreArchive(rng=random.Random(seed + 1))

        self._trajectory: list[Any] = []  # (action_id, data|None) since RESET
        self._prev_arr: Optional[np.ndarray] = None
        self._prev_hash: Optional[str] = None
        self._recent_hashes: deque[str] = deque(maxlen=loop_window)
        self._pending_restore: Optional[list[Any]] = None
        self._steps = 0

    # ---- public API --------------------------------------------------
    def act(
        self,
        frame: FrameDataRaw,
        legal_actions: Optional[list[GameAction]] = None,
    ) -> tuple[GameAction, Optional[dict[str, Any]]]:
        """Choose the next action given the latest observed frame.

        `legal_actions` should be `env.action_space` (i.e. derived from the
        current frame's `available_actions`); if omitted, it is read off
        `frame.available_actions` directly via `GameAction.from_id`, which
        is the verified-correct constructor (NOT `GameAction(n)`).
        """
        self._steps += 1

        if getattr(frame, "full_reset", False) or frame.state == GameState.GAME_OVER:
            self._on_episode_reset()
            return GameAction.RESET, None

        if frame.state == GameState.WIN:
            # Level cleared: reset local trajectory bookkeeping so the
            # archive/state-graph continue to track the *new* level's
            # states distinctly from the old one, then keep pushing RESET
            # is not required by the API on WIN — the harness advances the
            # level automatically on the next step — so we fall through to
            # normal action selection against the new frame.
            pass

        if legal_actions is None:
            legal_actions = [GameAction.from_id(a) for a in frame.available_actions]
        legal_ids = [a.value for a in legal_actions]
        if not legal_ids:
            # No legal actions reported yet (e.g. very first frame before
            # any RESET) -> RESET is always safe and always legal.
            return GameAction.RESET, None

        arr = self.codec.stack_to_array(frame.frame)
        frame_hash = self.codec.exact_hash(arr)
        node = self.graph.get_or_create(frame_hash, arr[0])

        pressure = self.pressure.transition_pressure(self._prev_arr, arr)
        self.archive.observe(frame_hash, self._trajectory, pressure)

        if self._prev_hash is not None:
            # record the edge for the action that produced *this* frame
            last_action_id, _ = self._trajectory[-1] if self._trajectory else (None, None)
            if last_action_id is not None:
                self.graph.record_edge(self._prev_hash, last_action_id, frame_hash)

        self._recent_hashes.append(frame_hash)
        self._prev_arr, self._prev_hash = arr, frame_hash

        # --- loop / no-op detection -------------------------------------
        looping = self._detect_loop()

        # --- decide: resume a pending Go-Explore restore, or explore ----
        if looping or self._pending_restore:
            action_id, data = self._restore_or_diversify(node, legal_ids, arr)
        else:
            action_id, data = self._choose_exploration_action(node, legal_ids, arr)

        self._trajectory.append((action_id, data))
        action = GameAction.from_id(action_id)
        return action, data

    # ---- internals -----------------------------------------------------
    def _on_episode_reset(self) -> None:
        self._trajectory = []
        self._prev_arr = None
        self._prev_hash = None
        self._recent_hashes.clear()
        self._pending_restore = None

    def _detect_loop(self) -> bool:
        if len(self._recent_hashes) < self._recent_hashes.maxlen:
            return False
        # A loop is: the same small set of hashes repeating with no growth
        # in distinct-state count over the window -> classic no-op/WAIT
        # cycling failure mode.
        distinct = len(set(self._recent_hashes))
        return distinct <= max(2, self._recent_hashes.maxlen // 4)

    def _choose_exploration_action(
        self, node: StateNode, legal_ids: list[int], arr: np.ndarray
    ) -> tuple[int, Optional[dict[str, Any]]]:
        untried = self.graph.untried_actions(node, legal_ids)
        if untried:
            action_id = self.rng.choice(untried)
        else:
            # Every legal action has been tried from this exact state at
            # least once: bias towards the action whose resulting state we
            # have visited least (or not recorded at all), i.e. maximise
            # expected new-state pressure rather than repeating the
            # historically-least-useful action.
            action_id = self._least_explored_edge(node, legal_ids)

        node.tried_actions.add(action_id)
        data = None
        if GameAction.from_id(action_id) in COMPLEX_ACTIONS:
            coord = self.pressure.best_click(arr[0], node.tried_clicks)
            node.tried_clicks.add(coord)
            data = {"y": coord[0], "x": coord[1]}
        return action_id, data

    def _least_explored_edge(
        self, node: StateNode, legal_ids: list[int]
    ) -> int:
        best_action, best_score = legal_ids[0], -1.0
        for action_id in legal_ids:
            dst = self.graph.edges.get((node.frame_hash, action_id))
            if dst is None:
                score = 1.0  # never observed a result for this action here
            else:
                dst_node = self.graph.nodes.get(dst)
                score = 1.0 / (1 + (dst_node.visits if dst_node else 0))
            if score > best_score:
                best_action, best_score = action_id, score
        return best_action

    def _restore_or_diversify(
        self, node: StateNode, legal_ids: list[int], arr: np.ndarray
    ) -> tuple[int, Optional[dict[str, Any]]]:
        """When looping, prefer an action never tried from the current
        node; if the current node is fully saturated, sample a Go-Explore
        restore target and walk its cached trajectory (replayed one step
        per call by consuming from `_pending_restore`) to break out of the
        local cycle into a different part of the state space.
        """
        if self._pending_restore:
            action_id, data = self._pending_restore.pop(0)
            if not self._pending_restore:
                self._pending_restore = None
            return action_id, data

        untried = self.graph.untried_actions(node, legal_ids)
        if untried:
            action_id = self.rng.choice(untried)
            node.tried_actions.add(action_id)
            data = None
            if GameAction.from_id(action_id) in COMPLEX_ACTIONS:
                coord = self.pressure.best_click(arr[0], node.tried_clicks)
                node.tried_clicks.add(coord)
                data = {"y": coord[0], "x": coord[1]}
            return action_id, data

        target = self.archive.select_restore_target()
        if target is not None and target.frame_hash != node.frame_hash:
            path = self.graph.shortest_path(node.frame_hash, target.frame_hash)
            if path:
                self._pending_restore = [(a, None) for a in path]
                action_id, data = self._pending_restore.pop(0)
                if not self._pending_restore:
                    self._pending_restore = None
                return action_id, data

        # Nothing better known: fall back to a uniformly random legal
        # action to guarantee forward progress rather than stalling.
        action_id = self.rng.choice(legal_ids)
        return action_id, None


## 3. Agent harness — real `arc_agi.Arcade` game loop
Matches the verified real API: `Arcade.make(game_id)`, `EnvironmentWrapper.reset/step/action_space/observation_space`, `Arcade.get_scorecard()`.

In [ ]:
import os

ARC_API_KEY = os.environ.get("ARC_API_KEY", "")  # anonymous key used if empty
GAME_IDS = os.environ.get("ARC_GAME_IDS", "ls20,ft09").split(",")
MAX_STEPS_PER_GAME = 5000  # hard ceiling so one game can never hang a submission


def run_game(arc: "arc_agi.Arcade", game_id: str) -> None:
    env = arc.make(game_id)
    if env is None:
        print(f"[skip] could not create environment for {game_id!r}")
        return

    agent = NoPriorSolverAgent(seed=0)
    frame = env.reset()
    if frame is None:
        print(f"[skip] reset() returned None for {game_id!r}")
        return

    steps = 0
    while steps < MAX_STEPS_PER_GAME:
        legal = env.action_space
        action, data = agent.act(frame, legal_actions=legal)
        frame = env.step(action, data=data)
        steps += 1
        if frame is None:
            break
        if frame.state == GameState.WIN and frame.levels_completed >= frame.win_levels:
            break

    print(f"[{game_id}] steps={steps} state={frame.state if frame else None} "
          f"levels_completed={getattr(frame, 'levels_completed', None)} "
          f"graph_nodes={len(agent.graph.nodes)}")


def main() -> None:
    arc = arc_agi.Arcade(arc_api_key=ARC_API_KEY)
    for game_id in GAME_IDS:
        game_id = game_id.strip()
        if not game_id:
            continue
        run_game(arc, game_id)
    print(arc.get_scorecard())


if __name__ == "__main__":
    main()


## 4. Notes for reviewers / open-source milestone submission
- **No prior**: zero pretrained model weights, zero network calls once a game is running, zero dataset-derived heuristics — every score in this notebook comes from `StructuralPressureScorer` computed live on observed frames.
- **Determinism**: `NoPriorSolverAgent(seed=...)` is fully seeded; identical environment behaviour reproduces identical trajectories, which matters for the Milestone open-source reproducibility requirement.
- **Extending toward more games**: the `StateGraph` / `GoExploreArchive` pair generalises across all ARC-AGI-3 games because it makes no game-specific assumption beyond the fixed 64x64/16-colour frame contract and the documented `GameAction` enum — this is why it is the recommended base to layer additional heuristics onto (e.g. game-specific object trackers) rather than replace.